# Benin — Exploratory Data Analysis (EDA)
**Objective:** Inspect, clean, and prepare Benin solar data for dashboarding.  
**Notes:** Save cleaned file to `../data/benin_clean.csv`. Do not commit `data/` to Git.

In [ ]:
# Cell: imports and basic config
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.figsize"] = (12, 4)


# Cell: load raw CSV (adjust path if needed)
df = pd.read_csv("../data/benin_raw.csv", parse_dates=["Timestamp"])
df.head(5)


# Cell: basic diagnostics
display(df.info())
display(df.describe(include="all").transpose())
display(df.isna().sum().sort_values(ascending=False))


# Cell: percent missing per column
missing_pct = df.isna().mean() * 100
missing_pct[missing_pct > 0].sort_values(ascending=False)
# highlight >5% for decision-making:
missing_pct[missing_pct > 5].sort_values(ascending=False)


# Cell: time range and frequency
print("Min timestamp:", df["Timestamp"].min())
print("Max timestamp:", df["Timestamp"].max())
# Check sample frequency (approx)
print("Total rows:", len(df))


# Cell: Z-score outliers for key columns
cols = ["GHI","DNI","DHI","ModA","ModB","WS","WSgust"]  # adjust if names differ
# keep only columns that exist
cols = [c for c in cols if c in df.columns]
z = np.abs(stats.zscore(df[cols].fillna(df[cols].median())))
outlier_mask = (z > 3).any(axis=1)  # any metric beyond |z|>3
print("Outlier rows detected:", outlier_mask.sum())
df_outliers = df[outlier_mask]
df_outliers.head()


# Cell: impute median for core columns
key_cols = ["GHI","DNI","DHI","ModA","ModB","Tamb"]
key_cols = [c for c in key_cols if c in df.columns]
for c in key_cols:
    median_val = df[c].median()
    df[c] = df[c].fillna(median_val)
# confirm
df[key_cols].isna().sum()


# Cell: remove outliers if you decided to
df_clean = df.loc[~outlier_mask].copy()
print("Rows before:", len(df), "after removing outliers:", len(df_clean))
# if you decide to keep outliers, set df_clean = df


# Cell: export cleaned file for use by Streamlit
# Save to data/ (project data folder). DO NOT commit this file to Git.
df_clean.to_csv("../data/benin_clean.csv", index=False)
print("Saved ../data/benin_clean.csv, rows:", len(df_clean))


# Cell: daily average GHI plot (matplotlib / seaborn)
df_clean['date'] = df_clean['Timestamp'].dt.date
daily = df_clean.groupby('date')['GHI'].mean().reset_index()
plt.figure(figsize=(14,4))
sns.lineplot(data=daily, x='date', y='GHI')
plt.xticks(rotation=45)
plt.title("Daily average GHI")
plt.show()


# Cell: boxplot comparisons (seaborn)
metrics = ["GHI","DNI","DHI"]  # customize to available columns
metrics = [m for m in metrics if m in df_clean.columns]
for m in metrics:
    plt.figure(figsize=(6,4))
    sns.boxplot(y=df_clean[m])
    plt.title(f"Boxplot — {m}")
    plt.show()


# Cell: correlation heatmap
corr_cols = ["GHI","DNI","DHI","TModA","TModB","Tamb","RH"]
corr_cols = [c for c in corr_cols if c in df_clean.columns]
plt.figure(figsize=(10,8))
sns.heatmap(df_clean[corr_cols].corr(), annot=True, cmap="vlag", fmt=".2f")
plt.title("Correlation matrix")
plt.show()


# Cell: wind rose (WS = wind speed, WD = wind direction degrees)
if ("WS" in df_clean.columns) and ("WD" in df_clean.columns):
    fig = px.scatter_polar(df_clean.sample(min(5000, len(df_clean))), r="WS", theta="WD", color="WS", opacity=0.6)
    fig.update_traces(marker=dict(size=3))
    fig.show()
else:
    print("No WS/WD columns available for wind rose.")


# Cell: cleaning effect if there is a Cleaning flag
if "Cleaning" in df_clean.columns:
    grouped = df_clean.groupby("Cleaning")[["ModA","ModB"]].mean().reset_index()
    display(grouped)
else:
    print("No 'Cleaning' column present.")


# Cell: load other cleaned country files and run ANOVA if available
import os
paths = {
    "Benin": "../data/benin_clean.csv",
    "Sierra": "../data/sierraleone_clean.csv",
    "Togo": "../data/togo_clean.csv"
}
dfs = {}
for k,p in paths.items():
    if os.path.exists(p):
        dfs[k] = pd.read_csv(p, parse_dates=["Timestamp"])
if len(dfs) >= 2 and "GHI" in list(dfs.values())[0].columns:
    samples = [dfs[k]['GHI'].dropna() for k in dfs]
    from scipy.stats import f_oneway
    fstat, pval = f_oneway(*samples)
    print("ANOVA F:", fstat, "p-value:", pval)
else:
    print("Need at least 2 cleaned country CSVs with GHI for ANOVA.")







## Short conclusion & recommendation

- Key findings (3 bullets about GHI mean/variance, obvious seasonality, cleaning effect).
- Data quality notes (missing columns, percent missing).
- Recommendation for MoonLight Energy Solutions:
  - Region X shows highest mean GHI and low variance → candidate for pilot.
  - Recommend more sensors or cleaning schedule in Region Y due to high variance.

